In [1]:
"""
ZaryahPlus Internship — Part A: Data Cleaning & SQL Storage
Author: Hamdaan
Dataset: Heart Disease UCI (Cleveland Clinic, 303 patients, 14 features)
 
What this script does:
  1. Loads the raw CSV using pandas
  2. Checks and handles missing values
  3. Removes duplicate rows
  4. Renames all columns to clean, readable names
  5. Saves the cleaned data into a SQLite database (heart_data.db)
  6. Runs 4 SQL queries and prints results """

'\nZaryahPlus Internship — Part A: Data Cleaning & SQL Storage\nAuthor: Hamdaan\nDataset: Heart Disease UCI (Cleveland Clinic, 303 patients, 14 features)\n \nWhat this script does:\n  1. Loads the raw CSV using pandas\n  2. Checks and handles missing values\n  3. Removes duplicate rows\n  4. Renames all columns to clean, readable names\n  5. Saves the cleaned data into a SQLite database (heart_data.db)\n  6. Runs 4 SQL queries and prints results '

In [2]:
import pandas as pd
import sqlite3
import os

In [3]:
# ─────────────────────────────────────────────
# STEP 1: Load the raw CSV
# ─────────────────────────────────────────────
# The original UCI Cleveland dataset uses '?' to represent missing values.


print("="*55)
print("Loading the Dataset")
print("="*55)

Loading the Dataset


In [4]:
pwd()

'C:\\Users\\phamd\\Zaryah Internal Screening'

In [5]:
# We tell pandas to treat '?' as NaN right from the start

df = pd.read_csv("C:\\Users\\phamd\\Zaryah Internal Screening\\heart_raw.csv", na_values=["?"])
 
print(f"Shape after loading: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
print(df.head())
 

Shape after loading: 222 rows × 14 columns

First 5 rows:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

   ca  thal  target  
0   0     6       0  
1   3     3       2  
2   2     7       1  
3   0     3       0  
4   0     3       0  


In [6]:
# STEP 2: Check and handle missing values
# ─────────────────────────────────────────────
# In the original UCI dataset, columns 'ca' and 'thal' have a small number
# of missing entries (marked as '?' in the raw file).
# Since they're only ~2% of rows, we drop those rows rather than impute —
# The model won't notice. But if 40% of a column were missing — dropping would be catastrophic, and then you'd be forced to impute.
# imputing categorical medical values without domain confirmation is risky.
""""
If thalassemia is missing, it means we don't know what the patient's status is. Filling it in with "3 = normal" doesn't make it normal — it makes a clinical assumption that could mislead the model. For continuous values like age or cholesterol, imputing with a mean is more defensible. For categorical clinical labels, it's riskier."""

'"\nIf thalassemia is missing, it means we don\'t know what the patient\'s status is. Filling it in with "3 = normal" doesn\'t make it normal — it makes a clinical assumption that could mislead the model. For continuous values like age or cholesterol, imputing with a mean is more defensible. For categorical clinical labels, it\'s riskier.'

In [7]:
print("\n" + "="*55)
print("Missing Value Check")
print("="*55)


Missing Value Check


In [8]:
missing_counts=df.isnull().sum()

print("missing counts per column")

print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "  No missing values found.")

missing counts per column
  No missing values found.


In [9]:
rows_before=len(df)
df.dropna(inplace=True)
rows_after=len(df)

print(f"\nRows before dropping NaN: {rows_before}")
print(f"Rows after  dropping NaN: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")


Rows before dropping NaN: 222
Rows after  dropping NaN: 222
Rows removed: 0


In [10]:
# STEP 3: Check and remove duplicate rows


In [11]:
print("\n" + "=" * 55)
print("STEP 3: Duplicate row check")
print("=" * 55)


STEP 3: Duplicate row check


In [12]:
duplicates=df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")


Duplicate rows found: 14


In [13]:
if duplicates>0:
    df.drop_duplicates(inplace=True)
    print(f"Duplicates removed.Rows remaining:{len(df)}")
else:
    print("No Duplicates.nothing to remove")

Duplicates removed.Rows remaining:208


In [14]:
# STEP 4: Rename columns to clean, readable names
# ─────────────────────────────────────────────
# The original column names are medical abbreviations.
# We rename them to plain English so any reader can understand the table.

In [15]:
print("\n" + "=" * 55)
print("STEP 4: Renaming columns")
print("=" * 55)


STEP 4: Renaming columns


In [16]:
column_rename_map = {
    "age":      "age",
    "sex":      "sex",
    "cp":       "chest_pain_type",
    "trestbps": "resting_blood_pressure",
    "chol":     "cholesterol",
    "fbs":      "fasting_blood_sugar",
    "restecg":  "resting_ecg",
    "thalach":  "max_heart_rate",
    "exang":    "exercise_induced_angina",
    "oldpeak":  "st_depression",
    "slope":    "st_slope",
    "ca":       "num_major_vessels",
    "thal":     "thalassemia",
    "target":   "heart_disease"
}
 
df.rename(columns=column_rename_map, inplace=True)
 
# The target column in the original dataset is 0 = no disease, 1–4 = disease.
# We convert it to a clean binary: 0 = no disease, 1 = disease.
# This is the standard approach used in all published studies on this dataset.
df["heart_disease"] = (df["heart_disease"] > 0).astype(int)
 
print("Columns after renaming:")
for original, renamed in column_rename_map.items():
    print(f"  {original:12s}  →  {renamed}")
 
print(f"\nFinal cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

Columns after renaming:
  age           →  age
  sex           →  sex
  cp            →  chest_pain_type
  trestbps      →  resting_blood_pressure
  chol          →  cholesterol
  fbs           →  fasting_blood_sugar
  restecg       →  resting_ecg
  thalach       →  max_heart_rate
  exang         →  exercise_induced_angina
  oldpeak       →  st_depression
  slope         →  st_slope
  ca            →  num_major_vessels
  thal          →  thalassemia
  target        →  heart_disease

Final cleaned dataset: 208 rows × 14 columns

Data types:
age                          int64
sex                          int64
chest_pain_type              int64
resting_blood_pressure       int64
cholesterol                  int64
fasting_blood_sugar          int64
resting_ecg                  int64
max_heart_rate               int64
exercise_induced_angina      int64
st_depression              float64
st_slope                     int64
num_major_vessels            int64
thalassemia                  int64

Why convert the target from 5 categories (0–4) to binary (0/1)?

The original values are: 0 = no disease, 1 = mild, 2 = moderate, 3 = severe, 4 = very severe. If you're a doctor and you want to screen patients — do you care how severe the disease is at the screening stage, or do you first just want to know does this patient have it or not?

That's exactly the reasoning. The task assigned to you says "binary target indicating whether the patient has heart disease or not." So:

0 = no disease → stays 0
1, 2, 3, 4 = disease present (varying severity) → all become 1

This also makes the ML task cleaner. A 5-class problem would need multi-class classifiers and much more data per class. A binary classifier is simpler, more interpretable, and directly answers the clinical question: "Should we investigate further?"


What is sqlite3?

SQLite3 is a lightweight, self-contained SQL database engine widely used in applications, mobile devices, and operating systems because it doesn’t require a separate server process.


In Python, the sqlite3 module provides a simple interface to work with SQLite databases directly from your code.

In [17]:
# STEP 5: Save to SQLite database
# ─────────────────────────────────────────────
# We create a SQLite file called heart_data.db.
# The cleaned DataFrame is written into a table called 'patients'.
# if_exists='replace' means we can safely re-run the script.

In [18]:
print("\n" + "=" * 55)
print("STEP 5: Saving to SQLite (heart_data.db → table: patients)")
print("=" * 55)


STEP 5: Saving to SQLite (heart_data.db → table: patients)


In [19]:
db_path = "heart_data.db"
conn = sqlite3.connect(db_path)
 
df.to_sql("patients", conn, if_exists="replace", index=False)
 
print(f"Data written to '{db_path}', table 'patients'.")
print(f"Rows in DB: {conn.execute('SELECT COUNT(*) FROM patients').fetchone()[0]}")
 

Data written to 'heart_data.db', table 'patients'.
Rows in DB: 208


In [20]:
#print("\n" + "=" * 55)
print("STEP 6: SQL Query Results")
print("=" * 55)
 
# ── Query 1: Count of patients with and without heart disease ──
print("\n── Query 1: Count of patients with / without heart disease ──")
query_1 = """
SELECT
    CASE WHEN heart_disease = 1 THEN 'Has Heart Disease'
         ELSE 'No Heart Disease' END AS status,
    COUNT(*) AS patient_count
FROM patients
GROUP BY heart_disease
ORDER BY heart_disease;
"""
result_1 = pd.read_sql_query(query_1, conn)
print(result_1.to_string(index=False))

STEP 6: SQL Query Results

── Query 1: Count of patients with / without heart disease ──
           status  patient_count
 No Heart Disease            127
Has Heart Disease             81


In [21]:
# ── Query 2: Average age grouped by heart disease status ──
print("\n── Query 2: Average age by heart disease status ──")
query_2 = """
SELECT
    CASE WHEN heart_disease = 1 THEN 'Has Heart Disease'
         ELSE 'No Heart Disease' END AS status,
    ROUND(AVG(age), 1) AS average_age
FROM patients
GROUP BY heart_disease
ORDER BY heart_disease;
"""


── Query 2: Average age by heart disease status ──


In [22]:
result_2 = pd.read_sql_query(query_2, conn)
result_2

,status,average_age
0,No Heart Disease,52.0
1,Has Heart Disease,57.3


In [23]:
print(result_2.to_string(index=False))

           status  average_age
 No Heart Disease         52.0
Has Heart Disease         57.3


In [24]:
# ── Query 3: Count by chest pain type, ordered descending ──
print("\n── Query 3: Patient count by chest pain type (descending) ──")
# Chest pain type values: 1=typical angina, 2=atypical angina,
# 3=non-anginal pain, 4=asymptomatic


── Query 3: Patient count by chest pain type (descending) ──


In [25]:
query_3 = """
SELECT
    CASE chest_pain_type
        WHEN 1 THEN '1 - Typical Angina'
        WHEN 2 THEN '2 - Atypical Angina'
        WHEN 3 THEN '3 - Non-Anginal Pain'
        WHEN 4 THEN '4 - Asymptomatic'
    END AS chest_pain_description,
    COUNT(*) AS patient_count
FROM patients
GROUP BY chest_pain_type
ORDER BY patient_count DESC;
"""
result_3 = pd.read_sql_query(query_3, conn)
result_3

,chest_pain_description,patient_count
0,4 - Asymptomatic,83
1,3 - Non-Anginal Pain,74
2,2 - Atypical Angina,41
3,1 - Typical Angina,10


In [26]:
print(result_3.to_string(index=False))

chest_pain_description  patient_count
      4 - Asymptomatic             83
  3 - Non-Anginal Pain             74
   2 - Atypical Angina             41
    1 - Typical Angina             10


In [27]:
# ── Query 4: Top 5 oldest patients with heart disease + cholesterol ──
print("\n── Query 4: Top 5 oldest patients with heart disease ──")
query_4 = """
SELECT
    age,
    sex,
    cholesterol,
    max_heart_rate,
    heart_disease
FROM patients
WHERE heart_disease = 1
ORDER BY age DESC
LIMIT 5;
"""
result_4 = pd.read_sql_query(query_4, conn)
result_4


── Query 4: Top 5 oldest patients with heart disease ──


,age,sex,cholesterol,max_heart_rate,heart_disease
0,71,1,322,109,1
1,68,1,277,151,1
2,67,1,286,108,1
3,67,1,229,129,1
4,67,1,254,163,1


In [28]:
print(result_4.to_string(index=False))


 age  sex  cholesterol  max_heart_rate  heart_disease
  71    1          322             109              1
  68    1          277             151              1
  67    1          286             108              1
  67    1          229             129              1
  67    1          254             163              1


In [29]:
# ── Close DB connection ──
conn.close()
 
print("\n" + "=" * 55)
print("Part A complete. Database saved as: heart_data.db")
print("Cleaned dataset shape:", df.shape)
print("=" * 55)


Part A complete. Database saved as: heart_data.db
Cleaned dataset shape: (208, 14)


In [30]:
# 